# 04 — Modeling: Baseline and Shortlist

The prepared **`train.csv`** is in the repository's `data/` folder; run the cells from top to bottom. The first code cell reads `../data/train.csv` directly. This notebook does not repeat the original data splitting or sampling and does not read `test.csv`.

Imputation, encoding, and scaling are fitted within each CV fold. The 2016 test sample remains reserved for final evaluation.

**Dependencies:** pandas, NumPy, scikit-learn >= 1.2, IPython, and threadpoolctl. Saved modeling results use the supplied preparation notebook's 150,000-loan training sample.

## 1. Modeling strategy

`Default = 1` represents a charged-off loan; `Default = 0` represents a fully paid loan. The FICO-only logistic regression establishes a simple reference. The five additional families are logistic regression, decision tree, random forest, Extra Trees, and histogram gradient boosting.

All models use identical stratified five-fold cross-validation splits within the 2007–2015 training sample. Imputation, encoding, and scaling are learned separately inside each fold. ROC-AUC is the primary ranking metric; average precision (AP), precision, recall, F1, accuracy, and Brier score are reported as mean ± sample standard deviation. Lower Brier scores are better; higher values are better for the other metrics.

Classification metrics use a preliminary threshold of 0.50. Cost-based threshold selection comes later. These shuffled historical folds do not simulate future deployment: the untouched 2016 test sample is reserved for final evaluation. The population also consists of completed, granted loans, so these results do not establish performance for all applicants.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, accuracy_score, brier_score_loss)

RANDOM_STATE = 42
TARGET = "Default"
# Read the training sample produced by the data preparation notebook.
train = pd.read_csv("../data/train.csv", low_memory=False)
model_data = train.copy().reset_index(drop=True)
print("Loaded train.csv.")

# Reject future-period data accidentally supplied as the training sample.
if "issue_d" in model_data:
    issue_dates = pd.to_datetime(model_data["issue_d"], errors="raise")
    assert issue_dates.dt.year.between(2007, 2015).all(), "Training sample includes loans outside 2007–2015."
if "id" in model_data:
    assert model_data["id"].is_unique, "Duplicate loan IDs in the training sample."

assert TARGET in model_data, "Default column is missing."
assert model_data[TARGET].notna().all(), "The target contains missing values."
assert set(model_data[TARGET].unique()) == {0, 1}, "Default must contain both classes, coded 0 and 1."
y = model_data[TARGET].astype(int)
print(f"Training rows: {len(y):,}; default rate: {y.mean():.2%}")

Loaded train.csv.
Training rows: 150,000; default rate: 18.46%


## 2. Application-time features and preprocessing

The code uses the supplied dataset's exact column names directly, without name normalization, lookup dictionaries, or automatic column matching. `fico_baseline` is copied from `fico_n` for use by both the baseline and full models.

Numeric inputs are `fico_baseline`, `revenue`, `dti_n`, and `loan_amnt`. Categorical inputs are `home_ownership_n`, `emp_length`, and `purpose`. Confirm the definitions and application-time availability of these inputs using the dataset documentation.

`experience_c` is excluded because its meaning is not documented here. Identifiers, issue dates, geographic columns, and free-text columns are excluded from this structured-feature comparison. DTI values are retained as supplied rather than assigned an undocumented missing-value code.

Numeric strings are converted to numbers, and infinities are replaced by missing values. All learned preprocessing stays inside each fold's Pipeline: median imputation and standardization for numeric inputs, and missing-category imputation plus one-hot encoding for categorical inputs.


In [2]:
def parse_numeric(series):
    return pd.to_numeric(series.astype("string").str.replace(",", "", regex=False)
                         .str.replace("%", "", regex=False), errors="coerce").astype(float)

# Use the supplied dataset's exact column names directly.
model_data["fico_baseline"] = parse_numeric(model_data["fico_n"])
assert model_data["fico_baseline"].notna().any(), "No valid FICO values found."

numeric_cols = ["fico_baseline", "revenue", "dti_n", "loan_amnt"]
categorical_cols = ["home_ownership_n", "emp_length", "purpose"]

for col in numeric_cols:
    model_data[col] = parse_numeric(model_data[col]).replace([np.inf, -np.inf], np.nan)
for col in categorical_cols:
    model_data[col] = model_data[col].map(lambda value: str(value) if pd.notna(value) else np.nan)
X = model_data[numeric_cols + categorical_cols].copy()
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
print("Excluded original columns:", [col for col in train.columns
      if col not in numeric_cols + categorical_cols and col != TARGET])

def make_preprocessor(num_cols, cat_cols):
    transformers = [("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler())]), num_cols)]
    if cat_cols:
        transformers.append(("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols))
    return ColumnTransformer(transformers)

preprocessor = make_preprocessor(numeric_cols, categorical_cols)
fico_preprocessor = make_preprocessor(["fico_baseline"], [])

Numeric features: ['fico_baseline', 'revenue', 'dti_n', 'loan_amnt']
Categorical features: ['home_ownership_n', 'emp_length', 'purpose']
Excluded original columns: ['id', 'issue_d', 'fico_n', 'experience_c', 'addr_state', 'zip_code', 'title', 'desc']


## 3. Baseline and stratified five-fold comparison

The following configurations are fixed starting points, not tuned models. Cross-validation produces one out-of-fold probability per training loan. Each loan is predicted by a model that did not train on that loan. Random forests and Extra Trees parallelize tree fitting; model families and folds run sequentially to limit memory use.

Tree fitting uses at most four workers. Native numerical thread pools are limited to four during modeling to avoid oversubscription; this changes computational resource use, not the CV design.

In [3]:
from threadpoolctl import threadpool_limits
thread_limits = threadpool_limits(limits=4)

In [4]:
estimators = {
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=14, min_samples_leaf=20,
                                            max_features="sqrt", n_jobs=4, random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=14, min_samples_leaf=20,
                                        max_features="sqrt", n_jobs=4, random_state=RANDOM_STATE),
    "Gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
        max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=1.0, random_state=RANDOM_STATE),
}
pipelines = {"FICO-only baseline": Pipeline([
    ("preprocess", fico_preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])}
for name, estimator in estimators.items():
    pipelines[name] = Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])

def calculate_metrics(y_true, probability):
    prediction = (probability >= 0.50).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y_true, probability),
        "AP": average_precision_score(y_true, probability),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
        "Accuracy": accuracy_score(y_true, prediction),
        "Brier": brier_score_loss(y_true, probability),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X, y))
fold_records, oof_probabilities, first_fold_models = [], {}, {}
for name, pipeline in pipelines.items():
    print(f"Training: {name}", flush=True)
    oof = np.full(len(X), np.nan)
    for fold_number, (fit_idx, valid_idx) in enumerate(folds, start=1):
        fitted = clone(pipeline)
        fitted.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        probability = fitted.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = probability
        print(f"  Fold {fold_number}/5: ROC-AUC={roc_auc_score(y.iloc[valid_idx], probability):.4f}", flush=True)
        fold_records.append({"Model": name, "Fold": fold_number,
                             **calculate_metrics(y.iloc[valid_idx], probability)})
        if fold_number == 1:
            first_fold_models[name] = fitted
    assert np.isfinite(oof).all()
    oof_probabilities[name] = oof

fold_results = pd.DataFrame(fold_records)
metric_names = ["ROC_AUC", "AP", "Precision", "Recall", "F1", "Accuracy", "Brier"]
cv_mean = fold_results.groupby("Model")[metric_names].mean()
cv_std = fold_results.groupby("Model")[metric_names].std()
ranking = cv_mean.sort_values(["ROC_AUC", "AP"], ascending=[False, False]).index
cv_report = pd.DataFrame(index=ranking)
for metric in metric_names:
    cv_report[metric] = [f"{cv_mean.loc[name, metric]:.4f} ± {cv_std.loc[name, metric]:.4f}"
                         for name in ranking]
display(cv_report)
baseline_auc = cv_mean.loc["FICO-only baseline", "ROC_AUC"]
baseline_ap = cv_mean.loc["FICO-only baseline", "AP"]
improvement = cv_mean[["ROC_AUC", "AP"]].copy()
improvement["ROC_AUC_gain"] = improvement["ROC_AUC"] - baseline_auc
improvement["AP_gain"] = improvement["AP"] - baseline_ap
display(improvement.loc[ranking].round(4))

Training: FICO-only baseline
  Fold 1/5: ROC-AUC=0.5840
  Fold 2/5: ROC-AUC=0.5909
  Fold 3/5: ROC-AUC=0.5887
  Fold 4/5: ROC-AUC=0.5952
  Fold 5/5: ROC-AUC=0.5931
Training: Logistic regression
  Fold 1/5: ROC-AUC=0.6525
  Fold 2/5: ROC-AUC=0.6502
  Fold 3/5: ROC-AUC=0.6524
  Fold 4/5: ROC-AUC=0.6552
  Fold 5/5: ROC-AUC=0.6596
Training: Decision tree
  Fold 1/5: ROC-AUC=0.6372
  Fold 2/5: ROC-AUC=0.6316
  Fold 3/5: ROC-AUC=0.6337
  Fold 4/5: ROC-AUC=0.6362
  Fold 5/5: ROC-AUC=0.6387
Training: Random forest
  Fold 1/5: ROC-AUC=0.6557
  Fold 2/5: ROC-AUC=0.6544
  Fold 3/5: ROC-AUC=0.6557
  Fold 4/5: ROC-AUC=0.6570
  Fold 5/5: ROC-AUC=0.6649
Training: Extra Trees
  Fold 1/5: ROC-AUC=0.6360
  Fold 2/5: ROC-AUC=0.6378
  Fold 3/5: ROC-AUC=0.6404
  Fold 4/5: ROC-AUC=0.6412
  Fold 5/5: ROC-AUC=0.6435
Training: Gradient boosting
  Fold 1/5: ROC-AUC=0.6601
  Fold 2/5: ROC-AUC=0.6569
  Fold 3/5: ROC-AUC=0.6580
  Fold 4/5: ROC-AUC=0.6590
  Fold 5/5: ROC-AUC=0.6679


,ROC_AUC,AP,Precision,Recall,F1,Accuracy,Brier
Model,,,,,,,
Gradient boosting,0.6604 ± 0.0044,0.2975 ± 0.0082,0.5383 ± 0.1973,0.0025 ± 0.0007,0.0050 ± 0.0014,0.8154 ± 0.0003,0.1430 ± 0.0005
Random forest,0.6575 ± 0.0042,0.2933 ± 0.0068,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1436 ± 0.0004
Logistic regression,0.6540 ± 0.0036,0.2899 ± 0.0064,0.5539 ± 0.0950,0.0026 ± 0.0005,0.0052 ± 0.0009,0.8155 ± 0.0002,0.1437 ± 0.0004
Extra Trees,0.6398 ± 0.0029,0.2789 ± 0.0070,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1459 ± 0.0002
Decision tree,0.6355 ± 0.0028,0.2695 ± 0.0035,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1453 ± 0.0002
FICO-only baseline,0.5904 ± 0.0043,0.2254 ± 0.0028,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1481 ± 0.0002


,ROC_AUC,AP,ROC_AUC_gain,AP_gain
Model,,,,
Gradient boosting,0.6604,0.2975,0.0700,0.0722
Random forest,0.6575,0.2933,0.0672,0.0679
Logistic regression,0.6540,0.2899,0.0636,0.0646
Extra Trees,0.6398,0.2789,0.0494,0.0535
Decision tree,0.6355,0.2695,0.0451,0.0442
FICO-only baseline,0.5904,0.2254,0.0000,0.0000


## 4. Held-out permutation importance

The top three non-baseline models by mean CV ROC-AUC form the provisional shortlist (AP breaks exact ties). Permutation importance measures the reduction in ROC-AUC after shuffling an original input feature through the complete fitted pipeline.

For computational efficiency, this diagnostic uses up to 3,000 rows from the first validation fold and five permutations per feature. The corresponding stored model did not train on these rows. The standard deviation describes permutation variability, not variability across folds. Correlated predictors can share importance; importance is predictive, not causal.

In [5]:
shortlist = [name for name in ranking if name != "FICO-only baseline"][:3]
print("Provisional shortlist:", shortlist)
_, importance_valid_idx = folds[0]
rng = np.random.default_rng(RANDOM_STATE)
importance_idx = rng.choice(importance_valid_idx, size=min(3000, len(importance_valid_idx)), replace=False)
assert y.iloc[importance_idx].nunique() == 2, "Importance sample must contain both classes."
importance_frames = []
for name in shortlist:
    result = permutation_importance(first_fold_models[name], X.iloc[importance_idx], y.iloc[importance_idx],
        scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
    table = pd.DataFrame({"Model": name, "Feature": X.columns,
                          "Importance_mean": result.importances_mean,
                          "Importance_std": result.importances_std}).sort_values("Importance_mean", ascending=False)
    importance_frames.append(table)
    print(f"\n{name}: top 10 features")
    display(table.head(10).round(4))
feature_importance_results = pd.concat(importance_frames, ignore_index=True)

Provisional shortlist: ['Gradient boosting', 'Random forest', 'Logistic regression']

Gradient boosting: top 10 features

Random forest: top 10 features

Logistic regression: top 10 features


,Model,Feature,Importance_mean,Importance_std
0,Gradient boosting,fico_baseline,0.0628,0.0060
3,Gradient boosting,loan_amnt,0.0619,0.0078
1,Gradient boosting,revenue,0.0327,0.0067
2,Gradient boosting,dti_n,0.0261,0.0049
6,Gradient boosting,purpose,0.0097,0.0044
4,Gradient boosting,home_ownership_n,0.0052,0.0016
5,Gradient boosting,emp_length,0.0034,0.0016


,Model,Feature,Importance_mean,Importance_std
0,Random forest,fico_baseline,0.0608,0.0052
3,Random forest,loan_amnt,0.0559,0.0069
2,Random forest,dti_n,0.0339,0.0052
1,Random forest,revenue,0.0304,0.0060
6,Random forest,purpose,0.0106,0.0029
4,Random forest,home_ownership_n,0.0088,0.0016
5,Random forest,emp_length,0.0025,0.0018


,Model,Feature,Importance_mean,Importance_std
0,Logistic regression,fico_baseline,0.0631,0.0072
3,Logistic regression,loan_amnt,0.0507,0.0051
2,Logistic regression,dti_n,0.0323,0.0031
1,Logistic regression,revenue,0.0180,0.0053
6,Logistic regression,purpose,0.0125,0.0051
5,Logistic regression,emp_length,0.0035,0.0013
4,Logistic regression,home_ownership_n,0.0034,0.0010


## 5. Which loans does the model get wrong?

At the 0.50 threshold, a **false negative** is a defaulted loan predicted as non-default; a **false positive** is a fully paid loan predicted as default. FNR = FN / actual defaults; FPR = FP / actual non-defaults.

Compare the shortlisted models, then inspect the top-ranked candidate across FICO bands, predicted-risk bands, loan categories, loan amounts, and DTI where available. Quartile bands are descriptive only and are not fed back into model training. Always examine group counts alongside rates; small segments can give unstable rates.

The individual-case tables show confident mistakes for investigation. Their presence alone does not demonstrate bad labels or explain why a borrower defaulted.

In [6]:
error_records = []
for name in shortlist:
    prediction = (oof_probabilities[name] >= 0.50).astype(int)
    actual = y.to_numpy()
    tn = int(((actual == 0) & (prediction == 0)).sum())
    fp = int(((actual == 0) & (prediction == 1)).sum())
    fn = int(((actual == 1) & (prediction == 0)).sum())
    tp = int(((actual == 1) & (prediction == 1)).sum())
    error_records.append({"Model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                          "FNR": fn / (fn + tp), "FPR": fp / (fp + tn)})
error_comparison = pd.DataFrame(error_records).set_index("Model")
display(error_comparison.round(4))

best_model_name = shortlist[0]
error_analysis = X.copy()
if "id" in model_data.columns:
    error_analysis.insert(0, "id", model_data["id"])
error_analysis["actual_default"] = y.to_numpy()
error_analysis["predicted_probability"] = oof_probabilities[best_model_name]
error_analysis["predicted_default"] = (error_analysis["predicted_probability"] >= 0.50).astype(int)
error_analysis["error_type"] = np.select([
    (error_analysis["actual_default"] == 1) & (error_analysis["predicted_default"] == 0),
    (error_analysis["actual_default"] == 0) & (error_analysis["predicted_default"] == 1)],
    ["False negative", "False positive"], default="Correct")
error_analysis["fico_band"] = pd.cut(error_analysis["fico_baseline"],
    bins=[-np.inf, 660, 700, 740, np.inf], labels=["<660", "660–699", "700–739", "740+"], right=False)
error_analysis["predicted_risk_band"] = pd.cut(error_analysis["predicted_probability"],
    bins=[0, 0.10, 0.20, 0.30, 0.50, 1],
    labels=["0–10%", "10–20%", "20–30%", "30–50%", "50–100%"], include_lowest=True)

def summarize_errors(data, group_col):
    working = data.copy()
    working["_fn"] = (working["error_type"] == "False negative").astype(int)
    working["_fp"] = (working["error_type"] == "False positive").astype(int)
    table = working.groupby(group_col, observed=True, dropna=False).agg(
        loans=("actual_default", "size"), defaults=("actual_default", "sum"),
        default_rate=("actual_default", "mean"), mean_predicted_risk=("predicted_probability", "mean"),
        false_negatives=("_fn", "sum"), false_positives=("_fp", "sum"))
    table["FNR"] = table["false_negatives"] / table["defaults"].replace(0, np.nan)
    table["FPR"] = table["false_positives"] / (table["loans"] - table["defaults"]).replace(0, np.nan)
    return table.round(4)

group_columns = ["fico_band", "predicted_risk_band"]
group_columns.extend(categorical_cols)
for col in ["loan_amnt", "dti_n", "revenue"]:
    if error_analysis[col].nunique() > 1:
        band_col = f"{col}_quartile"
        error_analysis[band_col] = pd.qcut(error_analysis[col], q=4, duplicates="drop")
        group_columns.append(band_col)
print(f"Detailed OOF error analysis: {best_model_name}")
error_group_tables = {}
for group_col in group_columns:
    print(f"\nErrors by {group_col}")
    error_group_tables[group_col] = summarize_errors(error_analysis, group_col)
    display(error_group_tables[group_col])
false_negatives = error_analysis.loc[error_analysis["error_type"] == "False negative"].sort_values("predicted_probability")
false_positives = error_analysis.loc[error_analysis["error_type"] == "False positive"].sort_values("predicted_probability", ascending=False)
print("\nDefaulted loans assigned the lowest predicted risk:")
display(false_negatives.head(10))
print("\nFully paid loans assigned the highest predicted risk:")
display(false_positives.head(10))

Detailed OOF error analysis: Gradient boosting

Errors by fico_band

Errors by predicted_risk_band

Errors by home_ownership_n

Errors by emp_length

Errors by purpose

Errors by loan_amnt_quartile

Errors by dti_n_quartile

Errors by revenue_quartile

Defaulted loans assigned the lowest predicted risk:

Fully paid loans assigned the highest predicted risk:


,TN,FP,FN,TP,FNR,FPR
Model,,,,,,
Gradient boosting,122244,72,27614,70,0.9975,0.0006
Random forest,122316,0,27684,0,1.0000,0.0000
Logistic regression,122255,61,27611,73,0.9974,0.0005


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
fico_band,,,,,,,,
<660,90,28,0.3111,0.2339,28,0,1.0000,0.0000
660–699,93317,20223,0.2167,0.2159,20153,69,0.9965,0.0009
700–739,42010,6179,0.1471,0.1481,6179,3,1.0000,0.0001
740+,14583,1254,0.0860,0.0885,1254,0,1.0000,0.0000


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
predicted_risk_band,,,,,,,,
0–10%,22972,1599,0.0696,0.0737,1599,0,1.0,0.0
10–20%,70646,10549,0.1493,0.1495,10549,0,1.0,0.0
20–30%,40625,9993,0.2460,0.2426,9993,0,1.0,0.0
30–50%,15615,5473,0.3505,0.3523,5473,0,1.0,0.0
50–100%,142,70,0.4930,0.5264,0,72,0.0,1.0


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
home_ownership_n,,,,,,,,
MORTGAGE,74067,12112,0.1635,0.1650,12112,4,1.0000,0.0001
OTHER,54,13,0.2407,0.1607,13,0,1.0000,0.0000
OWN,14863,2780,0.1870,0.1878,2779,0,0.9996,0.0000
RENT,61016,12779,0.2094,0.2075,12710,68,0.9946,0.0014


...

,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
loan_amnt_quartile,,,,,,,,
"(499.999, 8000.0]",40919,6017,0.1470,0.1489,6017,0,1.0000,0.0000
"(8000.0, 12000.0]",34379,6057,0.1762,0.1772,6057,0,1.0000,0.0000
"(12000.0, 20000.0]",42511,8672,0.2040,0.2039,8657,24,0.9983,0.0007
"(20000.0, 35000.0]",32191,6938,0.2155,0.2122,6883,48,0.9921,0.0019


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
dti_n_quartile,,,,,,,,
"(-0.001, 11.74]",37521,5164,0.1376,0.1392,5164,0,1.0000,0.0000
"(11.74, 17.48]",37496,6137,0.1637,0.1633,6136,0,0.9998,0.0000
"(17.48, 23.73]",37509,7221,0.1925,0.1933,7210,10,0.9985,0.0003
"(23.73, 83.4]",37474,9162,0.2445,0.2426,9104,62,0.9937,0.0022


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
revenue_quartile,,,,,,,,
"(1999.999, 45000.0]",38783,8161,0.2104,0.2094,8155,11,0.9993,0.0004
"(45000.0, 63500.0]",36240,7308,0.2017,0.2015,7275,39,0.9955,0.0013
"(63500.0, 90000.0]",39549,7153,0.1809,0.1798,7126,20,0.9962,0.0006
"(90000.0, 9500000.0]",35428,5062,0.1429,0.1454,5058,2,0.9992,0.0001


,id,fico_baseline,revenue,dti_n,loan_amnt,home_ownership_n,emp_length,purpose,actual_default,predicted_probability,predicted_default,error_type,fico_band,predicted_risk_band,loan_amnt_quartile,dti_n_quartile,revenue_quartile
44276,10153588,777.0,170000.0,8.55,8400.0,MORTGAGE,7 years,credit_card,1,0.017786,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]"
10921,1330328,792.0,90000.0,10.45,12000.0,MORTGAGE,10+ years,credit_card,1,0.020089,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(63500.0, 90000.0]"
3712,630666,807.0,90000.0,6.67,7000.0,MORTGAGE,NI,debt_consolidation,1,0.022527,0,False negative,740+,0–10%,"(499.999, 8000.0]","(-0.001, 11.74]","(63500.0, 90000.0]"
33505,6817730,777.0,135000.0,2.46,10000.0,MORTGAGE,10+ years,credit_card,1,0.023627,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]"
137549,64150409,812.0,92000.0,0.64,15000.0,MORTGAGE,10+ years,home_improvement,1,0.025452,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]"
134767,63516980,787.0,116000.0,16.90,16000.0,MORTGAGE,7 years,credit_card,1,0.027484,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(11.74, 17.48]","(90000.0, 9500000.0]"
82322,38614729,817.0,135000.0,8.59,18000.0,MORTGAGE,< 1 year,home_improvement,1,0.028790,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]"
66137,23864572,807.0,68000.0,9.18,14000.0,MORTGAGE,8 years,debt_consolidation,1,0.029225,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(63500.0, 90000.0]"
125287,60647501,777.0,95000.0,19.93,12000.0,MORTGAGE,8 years,credit_card,1,0.029512,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(17.48, 23.73]","(90000.0, 9500000.0]"
115007,55240369,767.0,95000.0,18.39,4000.0,MORTGAGE,10+ years,home_improvement,1,0.030028,0,False negative,740+,0–10%,"(499.999, 8000.0]","(17.48, 23.73]","(90000.0, 9500000.0]"


,id,fico_baseline,revenue,dti_n,loan_amnt,home_ownership_n,emp_length,purpose,actual_default,predicted_probability,predicted_default,error_type,fico_band,predicted_risk_band,loan_amnt_quartile,dti_n_quartile,revenue_quartile
90221,40530551,682.0,68000.0,36.80,23075.0,RENT,10+ years,small_business,0,0.663731,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]"
71089,32659627,662.0,68000.0,38.16,33000.0,RENT,10+ years,debt_consolidation,0,0.581748,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]"
21529,3527774,677.0,75010.0,30.00,25000.0,RENT,8 years,small_business,0,0.579372,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]"
110295,50920161,662.0,70000.0,38.21,24000.0,RENT,10+ years,debt_consolidation,0,0.572659,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]"
87458,39750752,662.0,62400.0,35.98,24975.0,RENT,< 1 year,debt_consolidation,0,0.572230,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(45000.0, 63500.0]"
89135,40450887,662.0,35000.0,30.76,15000.0,RENT,4 years,small_business,0,0.561460,1,False positive,660–699,50–100%,"(12000.0, 20000.0]","(23.73, 83.4]","(1999.999, 45000.0]"
5406,779272,702.0,85000.0,24.06,35000.0,RENT,2 years,small_business,0,0.560916,1,False positive,700–739,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]"
117907,55201657,682.0,60000.0,28.88,25200.0,RENT,9 years,moving,0,0.555463,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(45000.0, 63500.0]"
137475,64360949,682.0,91000.0,32.40,35000.0,RENT,10+ years,other,0,0.555097,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(90000.0, 9500000.0]"
7640,1039015,707.0,60000.0,21.34,27050.0,RENT,4 years,small_business,0,0.554817,1,False positive,700–739,50–100%,"(20000.0, 35000.0]","(17.48, 23.73]","(45000.0, 63500.0]"


## 6. Shortlist for the next modeling stage

The table below assembles the top three candidates' CV performance, gains over the FICO baseline, and out-of-fold error rates. These are provisional selections for further tuning. Review whether small performance gains justify complexity, and consider probability accuracy, stability, and operational error costs. A candidate can underperform the baseline; the gain columns make this visible.

Selection and diagnostic analysis reuse the historical CV results, so the winning CV score is not an unbiased final estimate. Final modeling choices and threshold selection must use training data only, followed by one final evaluation on the untouched 2016 test sample.

In [7]:
shortlist_summary = cv_report.loc[shortlist].copy()
shortlist_summary["ROC_AUC_gain_vs_baseline"] = (cv_mean.loc[shortlist, "ROC_AUC"] - baseline_auc).round(4)
shortlist_summary["AP_gain_vs_baseline"] = (cv_mean.loc[shortlist, "AP"] - baseline_ap).round(4)
shortlist_summary["OOF_FNR_at_0.50"] = error_comparison.loc[shortlist, "FNR"].round(4)
shortlist_summary["OOF_FPR_at_0.50"] = error_comparison.loc[shortlist, "FPR"].round(4)
display(shortlist_summary)
print("Models shortlisted for further tuning:")
for position, name in enumerate(shortlist, start=1):
    print(f"{position}. {name}")

Models shortlisted for further tuning:
1. Gradient boosting
2. Random forest
3. Logistic regression


,ROC_AUC,AP,Precision,Recall,F1,Accuracy,Brier,ROC_AUC_gain_vs_baseline,AP_gain_vs_baseline,OOF_FNR_at_0.50,OOF_FPR_at_0.50
Model,,,,,,,,,,,
Gradient boosting,0.6604 ± 0.0044,0.2975 ± 0.0082,0.5383 ± 0.1973,0.0025 ± 0.0007,0.0050 ± 0.0014,0.8154 ± 0.0003,0.1430 ± 0.0005,0.0700,0.0722,0.9975,0.0006
Random forest,0.6575 ± 0.0042,0.2933 ± 0.0068,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1436 ± 0.0004,0.0672,0.0679,1.0000,0.0000
Logistic regression,0.6540 ± 0.0036,0.2899 ± 0.0064,0.5539 ± 0.0950,0.0026 ± 0.0005,0.0052 ± 0.0009,0.8155 ± 0.0002,0.1437 ± 0.0004,0.0636,0.0646,0.9974,0.0005


## 7. Results discussion

The FICO-only baseline achieved a mean ROC-AUC of **0.5904 ± 0.0043** and mean average precision (AP) of **0.2254**. The training default rate was **18.46%**.

| Shortlisted model | CV ROC-AUC (mean ± std) | Mean AP | ROC-AUC gain over baseline |
|---|---:|---:|---:|
| Gradient boosting | 0.6604 ± 0.0044 | 0.2975 | 0.0700 |
| Random forest | 0.6575 ± 0.0042 | 0.2933 | 0.0672 |
| Logistic regression | 0.6540 ± 0.0036 | 0.2899 | 0.0636 |

All three improved discrimination over FICO alone. The differences between shortlisted models are small relative to their fold variability, so the ranking should be treated as provisional. Logistic regression remains a useful simpler candidate.

For gradient boosting, held-out permutation importance identified **FICO (0.0628 ROC-AUC reduction)** and **loan amount (0.0619)** as the strongest predictors, followed by revenue (0.0327) and DTI (0.0261). These results describe predictive reliance, not causal effects.

**The 0.50 classification threshold detects very few defaults.** Gradient boosting identified only **70 of 27,684 defaults** in the OOF predictions, missing 27,614 (FNR **99.75%**) and falsely flagging 72 performing loans. Random forest predicted no defaults at this threshold. This is a substantive limitation of the initial decision rule even though probability ranking improves over the baseline. Accuracy alone would conceal this problem. Cost-based threshold selection using training data is necessary before interpreting any model as a useful approval decision rule.

For gradient boosting, the FICO 660–699 segment contained **20,223 defaults across 93,317 loans** and had an FNR of **99.65%**. The 700–739 and 740+ segments had FNRs of **100%**. Small segments, such as FICO below 660 (90 loans), should not support strong comparisons. Among actual non-defaults, the highest DTI quartile had an FPR of **0.22%**, compared with **0%** in the lowest quartile. The very high FNR across segments primarily reflects the 0.50 threshold.

The training sample was originally produced by the supplied preparation notebook. This notebook now loads that saved `train.csv` directly. All six models completed five folds on the original **150,000-loan training sample**, followed by permutation importance and OOF error analysis. This modeling notebook does not read the 2016 test sample or use it for fitting, model selection, importance, or error analysis.

The shortlist proceeds to hyperparameter tuning and cost-based threshold selection using training data only. The 2016 test sample remains reserved for final evaluation. These saved results describe this execution and should be updated after changes to features or model configurations.
